# MASLD-LLM · 04 — Automated 1.7B Run (one-click / scheduled)



Top-to-bottom runner: **Runtime → Run all** does everything — mount, discover,

install, build the v5 mixture (class-balanced + geometric), train (with GPU

hygiene + skip-if-done guard), merge, and evaluate BOTH the raw (MB-20xx)

and geometric (MB-40xx) frozen test sets. No interactive input.



For unattended runs: **Colab Pro+ → Schedule** (clock icon next to Run all)

can execute this notebook at a chosen time with background execution.

All artifacts land on Drive and sync to the laptop, where the AutoClaw

autopilot scores them automatically.



Expected wall time on L4: ~10–14 h training + ~30 min eval. Interruptions

are safe (checkpoints auto-resume; re-Run-all skips completed stages).


## 1 · Mount + auto-discover


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']

def find_dir(target_name, search_roots=None, depth=6):
    if search_roots is None:
        search_roots = SEARCH_ROOTS
    matches = []
    for root in search_roots:
        direct = os.path.join(root, target_name)
        if os.path.isdir(direct):
            matches.append(direct)
    if matches:
        return matches
    def walk(path, d):
        if d > depth:
            return
        try:
            for entry in os.scandir(path):
                if entry.is_dir(follow_symlinks=False):
                    if entry.name == target_name:
                        matches.append(entry.path)
                    walk(entry.path, d + 1)
        except (PermissionError, OSError):
            pass
    for root in search_roots:
        if os.path.exists(root):
            walk(root, 0)
    return matches

cands = find_dir('masld-llm')
MASLD_LLM = min(cands, key=len) if cands else None
assert MASLD_LLM, 'masld-llm not found under Othercomputers/My Mac or MyDrive'
PROMPTS = os.path.join(MASLD_LLM, 'prompts')
print('PROMPTS =', PROMPTS)
print('corpora:', sorted(f for f in os.listdir(PROMPTS) if f.endswith('.jsonl.gz')))

def is_writable(path):
    try:
        probe = os.path.join(path, '.write_test')
        open(probe, 'w').close(); os.remove(probe)
        return True
    except Exception:
        return False

CKPT = (os.path.join(MASLD_LLM, 'checkpoints') if is_writable(MASLD_LLM)
        else '/content/drive/MyDrive/masld-llm-outputs/checkpoints')
os.makedirs(CKPT, exist_ok=True)
print('CKPT (writable) =', CKPT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROMPTS = /content/drive/Othercomputers/My Mac/masld-llm/prompts
corpora: ['MB-10xx_liver_train.jsonl.gz', 'MB-11xx_external_train.jsonl.gz', 'MB-20xx_liver_test.jsonl.gz', 'MB-30xx_dnam_test.jsonl.gz', 'MB-30xx_dnam_train.jsonl.gz', 'MB-61xx_msigdb_train.jsonl.gz', 'MB-62xx_drugs_train.jsonl.gz', 'MB-63xx_drugcentral_affinity_train.jsonl.gz', 'MB-70xx_dge_direction_train.jsonl.gz', 'MB-8xxx_fujiwara_train.jsonl.gz', 'MB-9xxx_fujiwara_KU_test.jsonl.gz']
CKPT (writable) = /content/drive/Othercomputers/My Mac/masld-llm/checkpoints


## 2 · Install LLaMA-Factory + convert corpora


In [ ]:
%cd /content
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip -q install -e ".[torch,metrics]" bitsandbytes peft transformers accelerate

import gzip, json, os

# --- Recovery block for missing global variables (PROMPTS) ---
if 'PROMPTS' not in globals():
    print("PROMPTS not found in workspace. Re-discovering environment...")
    SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']
    def find_dir(target_name, search_roots=None, depth=6):
        if search_roots is None:
            search_roots = SEARCH_ROOTS
        matches = []
        for root in search_roots:
            direct = os.path.join(root, target_name)
            if os.path.isdir(direct):
                matches.append(direct)
        if matches:
            return matches
        def walk(path, d):
            if d > depth:
                return
            try:
                for entry in os.scandir(path):
                    if entry.is_dir(follow_symlinks=False):
                        if entry.name == target_name:
                            matches.append(entry.path)
                        walk(entry.path, d + 1)
            except (PermissionError, OSError):
                pass
        for root in search_roots:
            if os.path.exists(root):
                walk(root, 0)
        return matches

    cands = find_dir('masld-llm')
    MASLD_LLM = min(cands, key=len) if cands else None
    assert MASLD_LLM, 'masld-llm not found under Othercomputers/My Mac or MyDrive. Please run the Mount cell first.'
    PROMPTS = os.path.join(MASLD_LLM, 'prompts')
    print('PROMPTS recovered =', PROMPTS)

def to_alpaca(in_path, out_path):
    rows = []
    with gzip.open(in_path, 'rt') as f:
        first = json.loads(f.readline())
        if '__meta__' not in first:
            rows.append(first)
        for l in f:
            rows.append(json.loads(l))
    out = [dict(instruction=r['prompt'], output=str(r['answer'])) for r in rows]
    json.dump(out, open(out_path, 'w'), indent=0)
    print(out_path, len(out))

to_alpaca(f'{PROMPTS}/MB-10xx_liver_train.jsonl.gz',      '/content/sft_liver.json')
to_alpaca(f'{PROMPTS}/MB-11xx_external_train.jsonl.gz',   '/content/sft_external.json')
to_alpaca(f'{PROMPTS}/MB-61xx_msigdb_train.jsonl.gz',     '/content/sft_knowledge.json')
to_alpaca(f'{PROMPTS}/MB-62xx_drugs_train.jsonl.gz',      '/content/sft_drugs.json')
to_alpaca(f'{PROMPTS}/MB-63xx_drugcentral_affinity_train.jsonl.gz', '/content/sft_drugs2.json')
to_alpaca(f'{PROMPTS}/MB-70xx_dge_direction_train.jsonl.gz', '/content/sft_dge.json')
to_alpaca(f'{PROMPTS}/MB-8xxx_fujiwara_train.jsonl.gz',   '/content/sft_fujiwara.json')
to_alpaca(f'{PROMPTS}/MB-30xx_dnam_train.jsonl.gz',       '/content/sft_dnam.json')

/content
fatal: destination path 'LLaMA-Factory' already exists and is not an empty directory.
/content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
/content/sft_liver.json 3928
/content/sft_external.json 5575
/content/sft_knowledge.json 2701
/content/sft_drugs.json 30000
/content/sft_drugs2.json 39632
/content/sft_dge.json 8088
/content/sft_fujiwara.json 204
/content/sft_dnam.json 73


## 3 · v5 mixture (class-balanced + geometric)


In [3]:
# ===== 1.7B RUN (v4 recipe) =====
# v3 results: pairwise 0.593 (beats frontier), multiclass 0.208 but collapsed to
# F0/F1 (early-stage bias from imbalanced upsampling), binary 0.405.
# v4: class-balanced multiclass (F2-F4 upweighted), binary yes upweighted,
# NAS upweighted. Model: Qwen3-1.7B. Expected train time ~10-14h on L4.
import os, json, random

# --- global recovery (if runtime was reset) ---
if 'CKPT' not in globals() or 'PROMPTS' not in globals():
    import glob as _glob
    SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']
    def find_dir(target_name, search_roots=None, depth=6):
        if search_roots is None: search_roots = SEARCH_ROOTS
        matches = []
        for root in search_roots:
            direct = os.path.join(root, target_name)
            if os.path.isdir(direct): matches.append(direct)
        if matches: return matches
        def walk(path, d):
            if d > depth: return
            try:
                for entry in os.scandir(path):
                    if entry.is_dir(follow_symlinks=False):
                        if entry.name == target_name: matches.append(entry.path)
                        walk(entry.path, d + 1)
            except (PermissionError, OSError): pass
        for root in search_roots:
            if os.path.exists(root): walk(root, 0)
        return matches
    _c = find_dir('masld-llm')
    MASLD_LLM = min(_c, key=len) if _c else None
    assert MASLD_LLM, 'masld-llm not found'
    PROMPTS = os.path.join(MASLD_LLM, 'prompts')
    CKPT = os.path.join(MASLD_LLM, 'checkpoints')
    print('recovered: MASLD_LLM =', MASLD_LLM)

rs = random.Random(20260929)

def load(p):
    return json.load(open(p)) if os.path.exists(p) else []

liver = load('/content/sft_liver.json')
external = load('/content/sft_external.json')
fujiwara = load('/content/sft_fujiwara.json')
knowledge = load('/content/sft_knowledge.json')
dge = load('/content/sft_dge.json')
drugs1 = load('/content/sft_drugs.json')
drugs2 = load('/content/sft_drugs2.json')
dnam = load('/content/sft_dnam.json')

def is_pairwise(ex):
    return 'Patient A:' in ex['instruction']
liver_single = [e for e in liver if not is_pairwise(e)]
liver_pair = [e for e in liver if is_pairwise(e)]
ext_single = [e for e in external if not is_pairwise(e)]
ext_pair = [e for e in external if is_pairwise(e)]

# --- v4: class-aware upsampling of liver singles ---
# multiclass answers: balance F0..F4 to ~equal (train dist F0:32 F1:50 F2:23 F3:23 F4:7)
STAGE_W = {'F0': 2, 'F1': 1, 'F2': 2, 'F3': 2, 'F4': 7}
def task_kind(ex):
    q = ex['instruction']
    if 'Assign the fibrosis stage' in q or 'Which fibrosis stage' in q: return 'multi'
    if 'advanced fibrosis' in q: return 'binary'
    if 'NAFLD Activity Score' in q or 'exact NAS' in q: return 'nas'
    if 'age in years' in q or 'chronological age' in q: return 'age'
    return 'fibreg'

def upsample_weighted(rows, base_factor, weight_fn):
    out = []
    for e in rows:
        w = weight_fn(e)
        n_copies = max(1, int(base_factor * w))
        out += [e] * n_copies
    return out

def liver_weight(ex):
    k = task_kind(ex)
    if k == 'multi':
        ans = str(ex['output']).strip()
        return STAGE_W.get(ans, 1)
    if k == 'binary':
        return 3 if str(ex['output']).strip().lower() == 'yes' else 1
    if k == 'nas':
        return 3
    return 1

liver_single_up = upsample_weighted(liver_single, 8, liver_weight)
fujiwara_up = list(fujiwara) * 8
dnam_up = list(dnam) * 20 if dnam else []

single = liver_single_up + ext_single + fujiwara_up + dnam_up
pair = liver_pair + ext_pair
drugs = rs.sample(drugs1 + drugs2, 10000)
know = rs.sample(knowledge + dge, 8000)

# --- v5: geometric train prompts (MB-40xx) join the mixture ---
def load_gz(p):
    import gzip as _gz
    rows = []
    if os.path.exists(p):
        with _gz.open(p, 'rt') as f:
            first = json.loads(f.readline())
            if '__meta__' not in first: rows.append(first)
            for l in f: rows.append(json.loads(l))
    return rows

geo = []
for root_candidate in ['/content', '/content/sft_geo.json']:
    pass
# convert the geo corpus from the prompts dir
import gzip as _gz2
geo_path = os.path.join(PROMPTS, 'MB-40xx_geo_train.jsonl.gz')
if os.path.exists(geo_path):
    with _gz2.open(geo_path, 'rt') as f:
        f.readline()
        for l in f:
            r = json.loads(l)
            geo.append(dict(instruction=r['prompt'], output=str(r['answer'])))
geo_up = geo * 6 if geo else []   # ~2.4k geometric examples
print(f'geometric prompts: {len(geo)} x6 -> {len(geo_up)}')

mixture = single + pair + drugs + know + geo_up
rs.shuffle(mixture)
json.dump(mixture, open('/content/sft_multitask_17b.json', 'w'), indent=0)
print(f'liver singles: {len(liver_single)} -> {len(liver_single_up)} (class-balanced x8 base)')
print(f'single pool {len(single)} | pair {len(pair)} | drugs {len(drugs)} | know {len(know)}')
print(f'FINAL v4 MIXTURE: {len(mixture)}')
from collections import Counter
mc = [str(e['output']) for e in liver_single_up if task_kind(e) == 'multi']
print('multiclass stage dist after balancing:', dict(sorted(Counter(mc).items())))
bn = [str(e['output']) for e in liver_single_up if task_kind(e) == 'binary']
print('binary yes/no dist:', dict(Counter(bn)))

geometric prompts: 405 x6 -> 2430
liver singles: 540 -> 7920 (class-balanced x8 base)
single pool 12015 | pair 7960 | drugs 10000 | know 8000
FINAL v4 MIXTURE: 40405
multiclass stage dist after balancing: {'F0': 512, 'F1': 400, 'F2': 368, 'F3': 368, 'F4': 392}
binary yes/no dist: {'yes': 720, 'no': 840}


## 4 · Training (GPU hygiene + skip-if-done)


In [ ]:
# 1.7B config + training (run AFTER the v4 sampler cell above)
import subprocess, sys, os

# ---- GPU hygiene: kill zombie processes holding VRAM from earlier cells ----
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
try:
    import torch
    del model  # drop any eval-cell model still referenced
except Exception:
    pass
try:
    import torch, gc
    gc.collect(); torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU free after cache clear: {free/1e9:.1f} / {total/1e9:.1f} GiB')
    if free < 12e9:
        # another process is holding VRAM — identify and kill it
        smi = subprocess.run(['nvidia-smi', '--query-compute-apps=pid,used_memory',
                              '--format=csv,noheader'], capture_output=True, text=True).stdout
        print('VRAM holders:\n' + smi)
        import os as _os
        my_pid = _os.getpid()
        for line in smi.strip().split('\n'):
            if not line.strip(): continue
            pid = int(line.split(',')[0].strip())
            if pid != my_pid:
                print(f'killing zombie GPU process {pid}...')
                subprocess.run(['kill', '-9', str(pid)])
        import time; time.sleep(5)
        free, total = torch.cuda.mem_get_info()
        print(f'GPU free after cleanup: {free/1e9:.1f} / {total/1e9:.1f} GiB')
        if free < 12e9:
            raise RuntimeError('GPU still occupied — use Runtime -> Restart session, then re-run cells 1, 2, 18, then THIS cell.')
except RuntimeError:
    raise
except Exception as e:
    print('(no torch/GPU yet:', e, ')')

ds_entry = '''{
  "masld_sft_17b": {
    "file_name": "/content/sft_multitask_17b.json",
    "columns": {"prompt": "instruction", "response": "output"}
  }
}'''
%cd /content/LLaMA-Factory
open('data/dataset_info.json', 'w').write(ds_entry)

CKPT_17B = f'{CKPT}/qwen3-1.7b-masld-sft'
yaml_17b = '''
model_name_or_path: Qwen/Qwen3-1.7B
trust_remote_code: true
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_alpha: 32
lora_dropout: 0.0
dataset: masld_sft_17b
template: qwen
cutoff_len: 4096
packing: true
neat_packing: true
max_samples: 100000
overwrite_cache: true
preprocessing_num_workers: 16
output_dir: ''' + CKPT_17B + '''
logging_steps: 10
save_steps: 0.25
plot_loss: true
per_device_train_batch_size: 1
gradient_accumulation_steps: 32
learning_rate: 1.0e-4
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
enable_liger_kernel: true
'''
open('/content/qwen17b_sft.yaml', 'w').write(yaml_17b)
print('config written; output:', CKPT_17B)

# CUDA + version fixes:
# - torchaudio must match torch's CUDA build
# - torchao must be ABSENT: with torchao installed, transformers-5.x loads
#   torchao-quantized linears and peft's LoRA dispatch then dies on an
#   incompatible torchao API (LinearActivationQuantizedTensor). Both observed
#   torchao failure modes vanish when it is simply not installed.
!pip uninstall -y torchaudio torchao
!pip install torchaudio --index-url https://download.pytorch.org/whl/cu130
!pip -q install liger-kernel
import importlib, subprocess, sys
if importlib.util.find_spec('torchao') is not None:
    print('WARNING: torchao still present - forcing removal')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
print('torchao absent: OK')

# ~10-14h on L4. Checkpoints save to Drive every quarter-epoch (resumable).
# --- automation guard: skip training if already complete ---
import json as _json
_SKIP_TRAIN = False
_done_marker = f'{CKPT}/qwen3-1.7b-masld-sft/train_results.json'
if os.path.exists(_done_marker):
    try:
        _res = _json.load(open(_done_marker))
        if float(_res.get('epoch', 0)) >= 2.0:
            print('Training already complete (>=2 epochs) - skipping to merge/eval')
            _SKIP_TRAIN = True
    except Exception:
        pass

if not _SKIP_TRAIN:
    import subprocess
    _r = subprocess.run([sys.executable, '-m', 'llamafactory.cli', 'train',
                         '/content/qwen17b_sft.yaml'],
                        env=dict(os.environ, DISABLE_VERSION_CHECK='1'))
    if _r.returncode != 0:
        raise RuntimeError('training failed - see llamafactory output above')
    print('training complete')

## 5 · Merge + dual evaluation (raw + geometric)


In [ ]:
# 1.7B merge + eval (run after training completes)
# Direct PeFT merge (skips the flaky llamafactory export), staleness-guarded,
# newline-stopped generation, UNIT-TESTED parser, archive + inline scores.
import os, glob, gzip, json, random, re, subprocess, sys, shutil, datetime
import numpy as np

CKPT_17B = f'{CKPT}/qwen3-1.7b-masld-sft'
MERGED_17B = '/content/qwen3-1.7b-masld-merged'

adapter = None
if os.path.exists(os.path.join(CKPT_17B, 'adapter_model.safetensors')):
    adapter = CKPT_17B
else:
    ckpts = sorted(glob.glob(f'{CKPT_17B}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[-1]))
    if ckpts: adapter = ckpts[-1]
assert adapter, f'no adapter in {CKPT_17B} - did training run?'
if adapter != CKPT_17B:
    print('NOTE: merging latest checkpoint (training interrupted?)')
print('merging adapter:', adapter)

if not os.path.exists(f'{MERGED_17B}/config.json') or \
   os.path.getmtime(f'{MERGED_17B}/model.safetensors' if os.path.exists(f'{MERGED_17B}/model.safetensors') else f'{MERGED_17B}/config.json') < os.path.getmtime(os.path.join(adapter, 'adapter_model.safetensors')):
    print('building fresh merge...')
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-1.7B', torch_dtype=torch.bfloat16)
    model = PeftModel.from_pretrained(base, adapter)
    model = model.merge_and_unload()
    model.save_pretrained(MERGED_17B, safe_serialization=True)
    AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B').save_pretrained(MERGED_17B)
    try:
        dst = f'{CKPT}/qwen3-1.7b-masld-merged'
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(MERGED_17B, dst)
        print('persisted to Drive')
    except Exception as e:
        print('Drive copy skipped:', e)
else:
    print('merge is fresh')

# --- eval with the TESTED parser ---
def parse_answer(text, task):
    raw = (text or '').strip()
    t = raw.lower()
    if task.endswith(('001', '201')):
        if t.startswith('yes') or 'answer: yes' in t or 'answer is yes' in t: return 'yes'
        if t.startswith('no') or 'answer: no' in t or 'answer is no' in t: return 'no'
        return None
    if task.endswith(('003', '203')):
        if re.search(r'\banswer(?: is|:) a\b', t) or t.startswith('a'): return 'A'
        if re.search(r'\banswer(?: is|:) b\b', t) or t.startswith('b'): return 'B'
        return None
    if task.endswith(('002', '202')):
        m = re.search(r'F([0-4])', raw)
        if m: return f'F{m.group(1)}'
        m = re.search(r'f([0-4])', t)
        return f'F{m.group(1)}' if m else None
    m = re.search(r'[Ff]([0-4])\b', raw)
    if m: return f'F{m.group(1)}'
    m = re.search(r'([0-9]+(?:\.[0-9]+)?)', t)
    if m:
        try: return float(m.group(1))
        except ValueError: return None
    return None

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MERGED_17B, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MERGED_17B, torch_dtype=torch.bfloat16, device_map='cuda')
model.eval()

rng = random.Random(20260929)
rows = []
with gzip.open(f'{PROMPTS}/MB-20xx_liver_test.jsonl.gz', 'rt') as f:
    f.readline()
    for line in f: rows.append(json.loads(line))
by_task = {}
for r in rows: by_task.setdefault(r['task'], []).append(r)
sample = []
for task, items in sorted(by_task.items()):
    rng.shuffle(items)
    sample.extend(items[:300])
print('eval prompts:', len(sample))

raws = []
BATCH = 8
i = 0
while i < len(sample):
    batch = sample[i:i+BATCH]
    try:
        enc = tok([r['prompt'] for r in batch], return_tensors='pt',
                  padding=True, truncation=True, max_length=4096).to('cuda')
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=30, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        for j in range(len(batch)):
            text = tok.decode(gen[j][enc['input_ids'].shape[1]:], skip_special_tokens=True)
            raws.append(text.split('\n')[0])
        i += len(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if BATCH == 1: raise
        BATCH = max(1, BATCH // 2)
        print(f'OOM -> batch {BATCH}')
    if i % 80 < BATCH:
        print(f'  {i}/{len(sample)}', flush=True)

runs_dir = (os.path.join(MASLD_LLM, 'results', 'frontier_runs')
            if os.access(os.path.join(MASLD_LLM, 'results'), os.W_OK)
            else '/content/drive/MyDrive/masld-llm-outputs/frontier_runs')
os.makedirs(runs_dir, exist_ok=True)
run_path = runs_dir + '/qwen3-1.7b-masld__MB-20xx.jsonl.gz'
with gzip.open(run_path, 'wt') as f:
    for r, raw in zip(sample, raws):
        f.write(json.dumps(dict(task=r['task'], sample=r['sample'], answer=r['answer'],
                                raw=raw, error=None, model_version='qwen3-1.7b-masld-v4',
                                parsed=parse_answer(raw, r['task']))) + chr(10))
print('archived ->', run_path)

import collections
bt = collections.defaultdict(list)
for r, raw in zip(sample, raws):
    bt[r['task']].append((parse_answer(raw, r['task']), r['answer']))
print('\n=== qwen3-1.7b-masld v4 ===')
for t, pairs in sorted(bt.items()):
    valid = sum(1 for p, g in pairs if p is not None)
    n = len(pairs)
    if t in ('MB-2001', 'MB-2002', 'MB-2003'):
        acc = sum(1 for p, g in pairs if p == g) / n
        print(f'  {t}: acc={acc:.3f} validity={valid/n:.3f}')
    else:
        nums = [(p, float(g)) for p, g in pairs if isinstance(p, (int, float))]
        if nums:
            mae = float(np.mean([abs(p-g) for p, g in nums]))
            from scipy.stats import spearmanr
            rho = float(spearmanr([g for p,g in nums],[p for p,g in nums]).statistic)
            print(f'  {t}: mae={mae:.3f} rho={rho:.3f} validity={valid/n:.3f}')
        else:
            print(f'  {t}: no parseable, validity={valid/n:.3f}')
print('\nreferences: 0.6B-v3 pairwise 0.593 / multi 0.208 / binary 0.405 | nemotron binary 0.661 / pair 0.500 / multi 0.231')

# ===== geometric test evaluation (the raw-vs-geometry ablation) =====
geo_rows = []
geo_path = os.path.join(PROMPTS, 'MB-40xx_geo_test.jsonl.gz')
if os.path.exists(geo_path):
    with gzip.open(geo_path, 'rt') as f:
        f.readline()
        for line in f:
            geo_rows.append(json.loads(line))
    print(f'\n=== geometric test: {len(geo_rows)} prompts (354 chars vs 4682 raw) ===')
    geo_raws = []
    BATCH = 16
    i = 0
    while i < len(geo_rows):
        batch = geo_rows[i:i+BATCH]
        try:
            enc = tok([r['prompt'] for r in batch], return_tensors='pt',
                      padding=True, truncation=True, max_length=1024).to('cuda')
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=30, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            for j in range(len(batch)):
                text = tok.decode(gen[j][enc['input_ids'].shape[1]:], skip_special_tokens=True)
                geo_raws.append(text.split('\n')[0])
            i += len(batch)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if BATCH == 1: raise
            BATCH = max(1, BATCH // 2)
    geo_run = runs_dir + '/qwen3-1.7b-masld__MB-40xx.jsonl.gz'
    with gzip.open(geo_run, 'wt') as f:
        for r, raw in zip(geo_rows, geo_raws):
            f.write(json.dumps(dict(task=r['task'], sample=r['sample'], answer=r['answer'],
                                    raw=raw, error=None, model_version='qwen3-1.7b-masld-v4-geo',
                                    parsed=parse_answer(raw, r['task']))) + chr(10))
    print('archived ->', geo_run)
    import collections as _c
    gt = _c.defaultdict(list)
    for r, raw in zip(geo_rows, geo_raws):
        gt[r['task']].append((parse_answer(raw, r['task']), r['answer']))
    print('=== qwen3-1.7b-masld GEOMETRIC (same model, same patients, 8% prompt length) ===')
    for t, pairs2 in sorted(gt.items()):
        valid = sum(1 for p, g in pairs2 if p is not None)
        n = len(pairs2)
        if t in ('MB-4001', 'MB-4002'):
            acc = sum(1 for p, g in pairs2 if p == g) / n
            print(f'  {t}: acc={acc:.3f} validity={valid/n:.3f}')
        else:
            nums = [(p, float(g)) for p, g in pairs2 if isinstance(p, (int, float))]
            if nums:
                mae = float(np.mean([abs(p-g) for p, g in nums]))
                from scipy.stats import spearmanr as _sp
                rho = float(_sp([g for p,g in nums],[p for p,g in nums]).statistic)
                print(f'  {t}: mae={mae:.3f} rho={rho:.3f} validity={valid/n:.3f}')
    print('\ngeometric-feature sanity refs: binary EN 0.679 / multi HGB 0.280 / reg MAE 1.210 rho 0.193')
else:
    print('(geometric test file not found - skipping ablation)')

## Automation notes

- Re-running the whole notebook after completion is cheap: training is

  skipped, merge is refreshed only if stale, eval re-runs (~30 min).

- If Colab interrupts mid-training: reconnect and Run all — training

  resumes from the last Drive checkpoint automatically.

- The AutoClaw autopilot on the laptop watches for the eval files and

  scores them; check `results/leaderboard.csv` and `RUNLOG.md`.
